In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
WORK = '/kaggle/working'

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
folds = pd.read_csv(find('folds.csv'))
fg = pd.read_csv(find('folds_group.csv'))
folds = folds.merge(fg[['uid', 'g120', 'g250']], on='uid', how='left')
assert folds[['g120', 'g250']].isna().sum().sum() == 0
te_uids = ['test/' + f for f in test.filename]
te_dur = pd.read_csv(find('audio_meta.csv')).set_index('uid').duration.reindex(te_uids).values
te_grp = np.where(te_dur < 40, 'short', np.where(te_dur < 52, 'dur_45s', 'dur_60s'))

TF = pd.read_csv(find('text_features.csv'), index_col='uid')
TF_tr, TF_te = TF.loc[folds.uid], TF.loc[te_uids]

LS = pd.read_csv(find('llm_scores.csv')).set_index('uid')
llm_cols = ['e', 'ent'] + [f'p{k}' for k in range(1, 6)] + [f'l{k}' for k in range(1, 6)]
LS_tr, LS_te = LS.loc[folds.uid, llm_cols], LS.loc[te_uids, llm_cols]

V = pd.read_csv(find('llm_views.csv')).set_index('uid')
v_ids = pd.read_csv(find('llm_views_uids.csv')).uid.tolist()
assert v_ids == V.index.tolist(), 'views uid order mismatch'
VIEWS = ('rubric', 'errors', 'wellformed', 'complexity')
v_cols = [f'{v}_{s}' for v in VIEWS for s in ['e', 'ent', 'p1', 'p2', 'p3', 'p4', 'p5']]
V_tr, V_te = V.loc[folds.uid, v_cols], V.loc[te_uids, v_cols]
MP = np.load(find('llm_rubric_meanpool.npy'), mmap_mode='r')
v_ix = {u: i for i, u in enumerate(v_ids)}
mp_tr = np.array([v_ix[u] for u in folds.uid])
mp_te = np.array([v_ix[u] for u in te_uids])
print('views:', V_tr.shape, '| mean-pooled hidden:', MP.shape, '| NaN in views:', int(V[v_cols].isna().sum().sum()))

y = folds.label.values
w = folds.w_test.values
nz = (folds.is_zero == 0).values
grp = folds.dur_group.values

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def pear(a, b): return float(np.corrcoef(a, b)[0, 1])
def wrmse(a, b, ww): return float(np.sqrt(np.sum(ww * (a - b) ** 2) / np.sum(ww)))
def wpear(a, b, ww):
    ma, mb = np.average(a, weights=ww), np.average(b, weights=ww)
    return float(np.sum(ww * (a - ma) * (b - mb)) /
                 np.sqrt(np.sum(ww * (a - ma) ** 2) * np.sum(ww * (b - mb) ** 2)))

def report(oof):
    r = {'non_zero': (rmse(y[nz], oof[nz]), pear(y[nz], oof[nz]))}
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (grp == g)
        r[g] = (rmse(y[m], oof[m]), pear(y[m], oof[m]))
    r['test_weighted'] = (wrmse(y[nz], oof[nz], w[nz]), wpear(y[nz], oof[nz], w[nz]))
    return pd.DataFrame(r, index=['RMSE', 'Pearson']).T.round(4)

def score(oof):
    r = report(oof)
    rm, pe = r.loc['test_weighted', 'RMSE'], r.loc['test_weighted', 'Pearson']
    return dict(rmse_w=rm, pear_w=pe, proxy=(rm + 1 - pe) / 2,
                pear_45=r.loc['dur_45s', 'Pearson'], pear_60=r.loc['dur_60s', 'Pearson'])

FOLD_COLS = ['g250', 'g120']
bo = pd.read_csv(find('base2_oof.csv')).set_index('uid').reindex(folds.uid)
bt = pd.read_csv(find('base2_test.csv')).set_index('filename').reindex(test.filename)
names0 = sorted({c.split('__')[0] for c in bo.columns})
R = {n: dict(oof={c: bo[f'{n}__{c}'].values for c in FOLD_COLS}, test=bt[n].values) for n in names0}
print('base models loaded from notebook 10:', names0)
W_, H_, T1, TL, TLn = 'wavlm_L21_ridge', 'whisper_L30_ridge', 'text_all_ridge', 'text_llm_ridge', 'textnoconf_llm_ridge'
LH = [n for n in names0 if n.startswith('llm_hidden_L')][0]

def avg_oof(names, col): return np.mean([R[n]['oof'][col] for n in names], axis=0)
def avg_test(names): return np.mean([R[n]['test'] for n in names], axis=0)

print('integrity avg|W+H+TLn proxy g250/g120 (expect 0.3732 / 0.3762):',
      round(score(avg_oof([W_, H_, TLn], 'g250'))['proxy'], 4), round(score(avg_oof([W_, H_, TLn], 'g120'))['proxy'], 4))
print('integrity avg|W+H+TLn+LH proxy g250/g120 (expect 0.3694 / 0.3740):',
      round(score(avg_oof([W_, H_, TLn, LH], 'g250'))['proxy'], 4), round(score(avg_oof([W_, H_, TLn, LH], 'g120'))['proxy'], 4))

In [ ]:
class Winsor(BaseEstimator, TransformerMixin):
    def __init__(self, lo=1, hi=99):
        self.lo, self.hi = lo, hi
    def fit(self, X, y=None):
        self.lo_ = np.percentile(X, self.lo, axis=0)
        self.hi_ = np.percentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lo_, self.hi_)

def ridge_text():
    return make_pipeline(SimpleImputer(strategy='median'), Winsor(), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-1, 4, 12)))

def ridge_hid():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 6, 11)))

def cv_oof(X, col, make_model):
    oof = np.full(len(y), np.nan)
    for f in range(5):
        va = (folds[col] == f).values
        m = make_model().fit(X[(~va) & nz], y[(~va) & nz])
        oof[va & nz] = np.clip(m.predict(X[va & nz]), 1, 5)
    return oof

def run(name, X_tr, X_te, make_model):
    X_tr, X_te = np.asarray(X_tr, dtype=np.float32), np.asarray(X_te, dtype=np.float32)
    oof = {c: cv_oof(X_tr, c, make_model) for c in FOLD_COLS}
    m = make_model().fit(X_tr[nz], y[nz])
    R[name] = dict(oof=oof, test=np.clip(m.predict(X_te), 1, 5))
    s, s2 = score(oof['g250']), score(oof['g120'])
    return dict(model=name, proxy_g250=s['proxy'], proxy_g120=s2['proxy'], rmse_w=s['rmse_w'],
                pear_w=s['pear_w'], pear_45=s['pear_45'], pear_60=s['pear_60'],
                train_rmse=rmse(y[nz], np.clip(m.predict(X_tr[nz]), 1, 5)))

cols = TF.columns.tolist()
conf = [c for c in cols if c.startswith(('wp_', 'seg_logprob', 'max_no_speech'))]
noconf = [c for c in cols if c not in conf]

rows = [run('views_ridge', V_tr.values, V_te.values, ridge_text),
        run('text_llm2_ridge', np.hstack([TF_tr[cols].values, LS_tr.values, V_tr.values]),
            np.hstack([TF_te[cols].values, LS_te.values, V_te.values]), ridge_text),
        run('textnoconf_llm2_ridge', np.hstack([TF_tr[noconf].values, LS_tr.values, V_tr.values]),
            np.hstack([TF_te[noconf].values, LS_te.values, V_te.values]), ridge_text)]

POOL_LAYERS = [6, 9, 12, 16, 20, 24]
sw = []
for j, L in enumerate(POOL_LAYERS):
    X = np.asarray(MP[mp_tr, j], dtype=np.float32)
    s1, s2 = score(cv_oof(X, 'g250', ridge_hid)), score(cv_oof(X, 'g120', ridge_hid))
    sw.append(dict(layer=L, j=j, proxy_mean=(s1['proxy'] + s2['proxy']) / 2, pear_w=s1['pear_w'],
                   rmse_w=s1['rmse_w'], pear_45=s1['pear_45'], pear_60=s1['pear_60']))
sw = pd.DataFrame(sw).round(4)
print('mean-pooled LLM hidden state by layer (last-token layer 9 from notebook 10 had mean proxy 0.4613):')
print(sw.drop(columns='j').to_string(index=False))
bj = int(sw.sort_values('proxy_mean').iloc[0].j)
bL = POOL_LAYERS[bj]
MPN = f'llm_meanpool_L{bL}_ridge'
rows.append(run(MPN, np.asarray(MP[mp_tr, bj], dtype=np.float32), np.asarray(MP[mp_te, bj], dtype=np.float32), ridge_hid))

tab = pd.DataFrame(rows).round(4)
print('\n', tab.sort_values('proxy_g250').to_string(index=False))

print('\ntrain (non-zero) vs test shift of the four expected scores:')
gtr = folds.dur_group.values
for c in [f'{v}_e' for v in VIEWS]:
    out = []
    for g in ['dur_45s', 'dur_60s']:
        a = V.loc[folds.uid[(gtr == g) & nz], c].values
        b = V.loc[np.array(te_uids)[te_grp == g], c].values
        out.append(f'{g}: train {a.mean():.3f} test {b.mean():.3f} KS p {ks_2samp(a, b).pvalue:.3f}')
    print(c.ljust(13), ' | '.join(out))

In [ ]:
def blend_cv(names, col, alpha=1.0):
    P = np.column_stack([R[n]['oof'][col] for n in names])
    out = np.full(len(y), np.nan)
    for f in range(5):
        va = (folds[col] == f).values
        m = Ridge(alpha=alpha, positive=True).fit(P[(~va) & nz], y[(~va) & nz])
        out[va & nz] = np.clip(m.predict(P[va & nz]), 1, 5)
    return out

def blend_fit(names, alpha=1.0, col='g250'):
    P = np.column_stack([R[n]['oof'][col] for n in names])
    T = np.column_stack([R[n]['test'] for n in names])
    m = Ridge(alpha=alpha, positive=True).fit(P[nz], y[nz])
    print('  weights', dict(zip(names, np.round(m.coef_, 3))), '| intercept', round(float(m.intercept_), 3))
    return np.clip(m.predict(T), 1, 5)

TL2, TLn2 = 'text_llm2_ridge', 'textnoconf_llm2_ridge'
sets = {'ref1: W+H+TL+LH': [W_, H_, TL, LH],
        'ref2: W+H+TLn+LH': [W_, H_, TLn, LH],
        'W+H+TL2+LH': [W_, H_, TL2, LH],
        'W+H+TLn2+LH': [W_, H_, TLn2, LH],
        'W+H+TL2+LH+MP': [W_, H_, TL2, LH, MPN],
        'W+H+TLn2+LH+MP': [W_, H_, TLn2, LH, MPN],
        'W+H+TL2+MP': [W_, H_, TL2, MPN],
        'W+H+TLn2+MP': [W_, H_, TLn2, MPN]}
cands = []
for sname, names in sets.items():
    for kind in ['avg', 'fit']:
        o = {c: (avg_oof(names, c) if kind == 'avg' else blend_cv(names, c)) for c in FOLD_COLS}
        s, s2 = score(o['g250']), score(o['g120'])
        cands.append(dict(name=f'{kind}|{sname}', kind=kind, n=len(names), names=names, oof=o,
                          proxy_g250=s['proxy'], proxy_g120=s2['proxy'], rmse_w=s['rmse_w'],
                          pear_w=s['pear_w'], pear_45=s['pear_45'], pear_60=s['pear_60']))
res = pd.DataFrame([{k: v for k, v in c.items() if k not in ('names', 'oof')} for c in cands])
res['proxy_mean'] = (res.proxy_g250 + res.proxy_g120) / 2
print(res.drop(columns=['kind', 'n']).round(4).sort_values('proxy_mean').to_string(index=False))

best = res.proxy_mean.min()
ok = res[res.proxy_mean <= best + 0.005].copy()
ok['is_fit'] = (ok.kind == 'fit').astype(int)
pick = ok.sort_values(['is_fit', 'n', 'proxy_mean']).iloc[0]
ch = next(c for c in cands if c['name'] == pick['name'])
bst = next(c for c in cands if c['name'] == res.sort_values('proxy_mean').iloc[0]['name'])
print('\nbest mean proxy:', round(best, 4), '| best candidate:', bst['name'], '| rule pick (simplest within 0.005):', ch['name'])
print(report(ch['oof']['g250']))

rng = np.random.RandomState(0)
idx_nz = np.where(nz)[0]
def proxy_on(o, ix):
    yy, pp, ww = y[ix], o[ix], w[ix]
    return (wrmse(yy, pp, ww) + 1 - wpear(yy, pp, ww)) / 2
def boot_diff(o_a, o_b, n=500):
    d = np.array([proxy_on(o_a, ix) - proxy_on(o_b, ix)
                  for ix in (rng.choice(idx_nz, len(idx_nz), replace=True) for _ in range(n))])
    return round(float(d.mean()), 4), round(float(np.percentile(d, 5)), 4), round(float(np.percentile(d, 95)), 4)

ref2 = next(c for c in cands if c['name'] == 'avg|ref2: W+H+TLn+LH')
print('\npaired bootstrap on g250 OOF, proxy of (candidate minus ref2), negative = candidate better. mean, 5th, 95th percentile:')
print('  rule pick :', boot_diff(ch['oof']['g250'], ref2['oof']['g250']))
print('  best      :', boot_diff(bst['oof']['g250'], ref2['oof']['g250']))
print('  (the bootstrap resamples rows, so it ignores cluster dependence and selection among candidates: intervals are too narrow)')

pred_te = avg_test(ch['names']) if ch['kind'] == 'avg' else blend_fit(ch['names'])
print('\ntest pred summary:', pd.Series(pred_te).describe().round(3).to_dict())
prev = avg_test([W_, H_, TLn])
print('corr with the notebook 10 chosen model test preds:', round(pear(pred_te, prev), 4))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
allp = res[['name', 'proxy_g250']].sort_values('proxy_g250')
ax[0].barh(allp.name, allp.proxy_g250)
ax[0].set_xlim(allp.proxy_g250.min() - 0.02, allp.proxy_g250.max() + 0.005)
ax[0].set_title('CV proxy on g250 folds (lower is better, axis truncated)')
sns.scatterplot(x=y[nz], y=ch['oof']['g250'][nz], hue=grp[nz], alpha=0.5, ax=ax[1])
ax[1].plot([1, 5], [1, 5], 'r--'); ax[1].set_xlabel('true'); ax[1].set_ylabel('OOF prediction')
ax[1].set_title('Rule pick: predicted vs true')
plt.tight_layout(); plt.show()

bp = pd.DataFrame({'uid': folds.uid})
for n in R:
    for c in FOLD_COLS:
        bp[f'{n}__{c}'] = R[n]['oof'][c]
bp.to_csv(f'{WORK}/base3_oof.csv', index=False)
pd.DataFrame({'filename': test.filename, **{n: R[n]['test'] for n in R}}).to_csv(f'{WORK}/base3_test.csv', index=False)
pd.DataFrame({'filename': test.filename, 'label': pred_te}).to_csv(f'{WORK}/submission_stack_v4.csv', index=False)
print('saved submission_stack_v4.csv, base3_oof.csv, base3_test.csv')